# COVID Symptoms Causal Analysis Using DEMI

Loads the cleaned COVIDCARE dataset and DEMI knowledgebase, calculates symptom evidence for PCR-positive COVID-19, and runs a demonstration patient scenario.

In [ ]:
import pandas as pd
import numpy as np
import math
import matplotlib.pyplot as plt
from pathlib import Path

raw_path = Path('COVIDCARE_FORSUBMISSION_MIT_CLEANED_Phase_II_2021-12-03.csv')
kb_path = Path('COVIDCARE_DEMI_knowledgebase_v4.csv')
out_dir = Path('outputs')
out_dir.mkdir(exist_ok=True)

In [ ]:
raw = pd.read_csv(raw_path)
kb = pd.read_csv(kb_path)
print('Raw dataset:', raw.shape)
print('Knowledgebase:', kb.shape)
raw['PCR Test Positive'].value_counts(dropna=False)

## Prepare PCR-positive target evidence

In [ ]:
pcr = kb[kb['target_concept_code'] == 'target_pcr_positive'].copy()
a = pcr['n_code_target'].astype(float)
b = pcr['n_code_no_target'].astype(float)
c = pcr['n_target_no_code'].astype(float)
d = pcr['n_no_code_no_target'].astype(float)

pcr['p_target_given_code'] = (a + 0.5) / (a + b + 1)
pcr['p_target_given_no_code'] = (c + 0.5) / (c + d + 1)
pcr['risk_diff'] = pcr['p_target_given_code'] - pcr['p_target_given_no_code']
pcr['odds_ratio'] = ((a + 0.5) * (d + 0.5)) / ((b + 0.5) * (c + 0.5))
pcr['LR_pos'] = ((a + 0.5) / (a + c + 1)) / ((b + 0.5) / (b + d + 1))
pcr['temporal_ratio'] = (pcr['n_code_before_target'] + 1) / (pcr['n_target_before_code'] + 1)
pcr['symptom'] = pcr['standard_concept_name'].str.extract(r';\s*(.*)$')[0].fillna(pcr['standard_concept_name'])
pcr.head()

In [ ]:
baseline = raw['PCR Test Positive'].mean()
known = raw['PCR Test Positive'].notna().sum()
pos = int(raw['PCR Test Positive'].sum())
neg = int(known - pos)
print(f'Known PCR outcomes: {known}; positive: {pos}; negative: {neg}; baseline={baseline:.3%}')

## Symptoms with the highest PCR-positive probability

In [ ]:
symptoms = pcr[pcr['standard_concept_name'].str.contains('In the last 14 days', case=False, na=False)].copy()
symptoms = symptoms[~symptoms['symptom'].str.contains('None|Prefer not', case=False, na=False)]
symptoms = symptoms[~symptoms['standard_concept_name'].str.contains('traveled', case=False, na=False)]
top = symptoms.sort_values('p_target_given_code', ascending=False).head(12)
display(top[['concept_code','symptom','n_code_target','n_code_no_target','p_target_given_code','risk_diff','odds_ratio','LR_pos']])
top.to_csv(out_dir/'top_symptoms_summary.csv', index=False)

In [ ]:
plot_top = top.copy().sort_values('p_target_given_code')
plt.figure(figsize=(9,5.5))
plt.barh(plot_top['symptom'], plot_top['p_target_given_code']*100)
plt.xlabel('Probability of PCR-positive result when symptom is present (%)')
plt.title('Most Informative Symptoms in the Knowledgebase')
plt.tight_layout()
plt.savefig(out_dir/'top_symptoms.png', dpi=200)
plt.show()

## Patient scenario: symptom progression
Scenario: fever, cough, fatigue, headache, loss of smell, and loss of taste.

In [ ]:
scenario_names = ['Fever or feeling feverish','Cough','Fatigue (more than normal)','Headaches','Loss of smell','Loss of taste']
scenario_rows = []
for name in scenario_names:
    cand = pcr[(pcr['symptom'].eq(name)) & pcr['standard_concept_name'].str.contains('last 14 days', case=False, na=False)]
    scenario_rows.append(cand.iloc[0])
scenario = pd.DataFrame(scenario_rows)
display(scenario[['concept_code','symptom','p_target_given_code','risk_diff','LR_pos']])
scenario.to_csv(out_dir/'scenario_symptoms_summary.csv', index=False)

In [ ]:
scenario_codes = ['30153-Symptoms-2','30154-Symptoms_Resp-3','30153-Symptoms-5','30158-Symtpom_Neuro-2','30158-Symtpom_Neuro-7','30158-Symtpom_Neuro-8']
df = raw[raw['PCR Test Positive'].notna()].copy()
progress = [{'Step':'Baseline','N':len(df),'Positive':int(df['PCR Test Positive'].sum()),'Empirical probability':df['PCR Test Positive'].mean()}]
labels = ['Fever','Fever + cough','+ fatigue','+ headache','+ loss of smell','+ loss of taste']
for i,label in enumerate(labels, start=1):
    m = df[scenario_codes[:i]].fillna(0).eq(1).all(axis=1)
    n = int(m.sum())
    pos = int(df.loc[m, 'PCR Test Positive'].sum()) if n else 0
    progress.append({'Step':label,'N':n,'Positive':pos,'Empirical probability':pos/n if n else np.nan})
progress_df = pd.DataFrame(progress)
display(progress_df)
progress_df.to_csv(out_dir/'scenario_progression.csv', index=False)

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(progress_df['Step'], progress_df['Empirical probability']*100, marker='o')
plt.ylabel('Observed PCR-positive probability (%)')
plt.title('Patient Scenario: Symptom Progression and Estimated Risk')
plt.xticks(rotation=30, ha='right')
for i,row in progress_df.iterrows():
    plt.text(i, row['Empirical probability']*100+2, f"n={int(row['N'])}", ha='center', fontsize=8)
plt.tight_layout()
plt.savefig(out_dir/'scenario_progression.png', dpi=200)
plt.show()

## DEMI-style final probability

In [ ]:
kb_baseline = (pcr.iloc[0]['n_code_target'] + pcr.iloc[0]['n_target_no_code']) / (pcr.iloc[0]['n_code_target'] + pcr.iloc[0]['n_code_no_target'] + pcr.iloc[0]['n_target_no_code'] + pcr.iloc[0]['n_no_code_no_target'])
logit = math.log(kb_baseline/(1-kb_baseline)) + 0.5 * sum(math.log(float(x)) for x in scenario['LR_pos'])
final_probability = 1 / (1 + math.exp(-logit))
print(f'Knowledgebase baseline: {kb_baseline:.1%}')
print(f'Final DEMI-style probability for full scenario: {final_probability:.1%}')

## Interpretation
A high probability supports testing, isolation precautions, and follow-up, but the model does not diagnose COVID-19. In real use, baseline probability should be calibrated to current local COVID-19 prevalence.